# Problem 4: Image Classifier

**Objective:** build a web application that identifies the contents of a photo, chosen from a list of categories defined by the team. No training data and no API key are required.

This project uses **CLIP**, an open-weight model that learned to associate images with text by studying 400 million image-caption pairs. Because it understands text, it can be given **any list of labels** and will select the best match. This is called **zero-shot classification**: the model has not been specifically trained on the team's labels.

**Learning outcomes:** using a pre-trained open model from Hugging Face, understanding how label wording affects AI performance, and measuring accuracy.

### How to use this notebook
- A notebook is made up of **cells**. To run a cell, select it and press **Shift + Enter** (or select the run button).
- On the first run, run the cells **in order, from top to bottom**.
- Comments marked **STUDENT TODO** show where changes are expected:
  - **Level 1:** text changes. No coding experience required.
  - **Level 2:** changes to settings and values.
  - **Level 3:** new features. Mentors are available to help.
- After changing a cell, **re-run that cell and then re-run the "Launch the application" cell** for the change to take effect.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

> **Note:** the AI model in this project runs **within Google Colab**, so no API key is required.
> Optional: for faster processing, select **Runtime → Change runtime type → T4 GPU**. The notebook also works without a GPU.

## Step 0: Installation and model loading (run both cells)

In [ ]:
# Installs the required libraries. This takes about 30 seconds.
!pip install -q "gradio>=5,<6" transformers pillow requests

In [ ]:
# Downloads the open-weight CLIP model from Hugging Face (about 600 MB; about one minute on the first run).
import torch
from transformers import pipeline

# STUDENT TODO (Level 3): try a different open-weight model and compare accuracy.
#   "openai/clip-vit-base-patch32"   CLIP Base (default, fast)
#   "openai/clip-vit-large-patch14"  CLIP Large (more accurate, about 1.7 GB, slower)
#   "google/siglip-base-patch16-224" Google SigLIP (a newer approach)
MODEL_NAME = "openai/clip-vit-base-patch32"

device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-image-classification", model=MODEL_NAME, device=device)
print(f"Loaded {MODEL_NAME} on {'GPU' if device == 0 else 'CPU'}.")

## Step 1: Define the theme and labels

Select a theme for the classifier. Examples: regional foods, recyclable and non-recyclable items, dog breeds, campus locations, facial expressions, plant types, footwear.

**Guidance:** CLIP performs best with **clear, specific and visual** labels. For example, "golden retriever" performs better than "friendly dog".

In [ ]:
# STUDENT TODO (Level 1): set the application name and description.
APP_TITLE = "Animal Identifier"
APP_DESCRIPTION = "Upload a photo or use the webcam. An open-weight AI model identifies the animal shown."

# STUDENT TODO (Level 1): the list of categories. Add, remove or change labels as needed.
LABELS = [
    "cat",
    "dog",
    "rabbit",
    "parrot",
    "fish",
    "human",
]

# STUDENT TODO (Level 2): the sentence template used to present each label to the model.
#   {} is replaced by each label. Alternatives:
#   "a photo of a {}."   "a blurry photo of a {}."   "a close-up photo of {}, a type of food."
#   The template can noticeably affect accuracy and should be tested.
LABEL_TEMPLATE = "a photo of a {}."

# STUDENT TODO (Level 2): if the top result's confidence (0 to 1) is below this value, report it as uncertain.
CONFIDENCE_THRESHOLD = 0.40

# STUDENT TODO (Level 3): additional information shown for each label when the model is confident.
LABEL_NOTES = {
    "cat": "Cats spend about 70% of their lives sleeping.",
    "dog": "Each dog's nose print is unique, similar to a human fingerprint.",
    "parrot": "Some parrot species can learn more than 100 words.",
}

print(f"{len(LABELS)} labels configured: {LABELS}")

## Step 2: The classification function (run this cell)

This function takes an image and a list of labels, asks the model how well each label matches the image, and prepares the results.

In [ ]:
from PIL import Image

def classify(image, labels_text):
    if image is None:
        return None, "Upload a photo or capture one with the webcam."

    # Convert the comma-separated text into a list of labels.
    labels = [label.strip() for label in labels_text.split(",") if label.strip()]
    if len(labels) < 2:
        return None, "Enter at least two labels, separated by commas."

    # Ask the model how well each label matches the image.
    results = classifier(image, candidate_labels=labels, hypothesis_template=LABEL_TEMPLATE)
    scores = {r["label"]: r["score"] for r in results}

    # Prepare a message describing the top result.
    top = results[0]
    if top["score"] < CONFIDENCE_THRESHOLD:
        verdict = f"Result uncertain. Most likely: **{top['label']}** ({top['score']:.0%} confidence)."
    else:
        verdict = f"### Result: **{top['label']}** ({top['score']:.0%} confidence)"
        if top["label"] in LABEL_NOTES:
            verdict += f"\n\n{LABEL_NOTES[top['label']]}"

    # STUDENT TODO (Level 3): add custom logic here.
    #   Example for a recycling application:
    #   if top["label"] in ["plastic bottle", "aluminium can"]: verdict += "\n\nDispose of in the recycling bin."
    return scores, verdict


# Quick test using a sample photo from the internet.
import requests
from IPython.display import display

SAMPLE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"  # STUDENT TODO (Level 1): any image URL may be used
try:
    sample = Image.open(requests.get(SAMPLE_URL, stream=True, timeout=15).raw).convert("RGB")
    sample.thumbnail((320, 320))
    display(sample)
    scores, verdict = classify(sample, ", ".join(LABELS))
    print(verdict.replace("#", "").replace("*", "").strip(), "\n")
    for label, score in scores.items():
        print(f"{label:>12} {'█' * round(score * 30):<30} {score:.0%}")
except Exception as e:
    print("The sample image could not be downloaded. The application below will still work.", e)

## Step 3: Launch the application

Run this cell. The application appears below, and a **public link** is also created, which can be opened on a mobile phone for testing with the phone camera.

Labels can be **edited directly in the application** for quick experiments. When a set of labels performs well, copy it back into `LABELS` above.

In [ ]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running

with gr.Blocks(theme=gr.themes.Soft(), title=APP_TITLE) as demo:  # STUDENT TODO (Level 2): alternative theme: gr.themes.Glass()
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", sources=["upload", "webcam", "clipboard"], label="Photo")
            labels_input = gr.Textbox(value=", ".join(LABELS), label="Labels (comma-separated; editable)")
            button = gr.Button("Classify", variant="primary")
        with gr.Column():
            verdict_output = gr.Markdown()
            scores_output = gr.Label(num_top_classes=5, label="Top results")  # STUDENT TODO (Level 2): number of results shown

    button.click(classify, inputs=[image_input, labels_input], outputs=[scores_output, verdict_output])
    image_input.change(classify, inputs=[image_input, labels_input], outputs=[scores_output, verdict_output])

demo.launch(share=True)

## Level 3: Optional extensions

**1. Accuracy measurement (recommended for the demonstration).** Collect **10 test photos** for the chosen theme and record how many are classified correctly. Refine `LABELS` and `LABEL_TEMPLATE`, then measure again. Present the results before and after refinement.

**2. Model comparison.** Change `MODEL_NAME` to `"openai/clip-vit-large-patch14"` or `"google/siglip-base-patch16-224"`, re-run from Step 0, and repeat the 10-photo test. Assess whether the larger model justifies the extra processing time.

**3. Limitations.** Identify the conditions under which the model performs poorly, for example drawings, unusual angles, or several objects in one image. Record examples for the demonstration.

**4. Multiple themes.** Add a menu to switch between label sets.
<details><summary>Hint (select to expand)</summary>

```python
THEMES = {
    "Animals": "cat, dog, rabbit, parrot, fish",
    "Food": "pizza, burger, samosa, dosa, salad",
}
theme_dropdown = gr.Dropdown(list(THEMES), value="Animals", label="Theme")
# When the selection changes, update the labels field:
theme_dropdown.change(lambda t: THEMES[t], inputs=theme_dropdown, outputs=labels_input)
```
</details>

**5. Practical application.** For example, a waste-sorting application: classify an item, then map each label to "Recycling", "Compost" or "General waste" using a dictionary similar to `LABEL_NOTES`.

## Demonstration checklist
- [ ] A live demonstration using a webcam or phone camera
- [ ] Accuracy results on 10 test photos, before and after refining the labels
- [ ] The team can explain zero-shot classification in one sentence
- [ ] At least one example of an incorrect classification

## Troubleshooting

| Issue | Resolution |
|---|---|
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| The application does not reflect recent changes | Re-run the edited cell **and** the "Launch the application" cell. |
| The notebook remains in demonstration mode | Add `GROQ_API_KEY` in Secrets, enable **Notebook access**, and re-run the setup cell. |
| "Usage limit reached" error | Wait about one minute. Free keys allow a limited number of requests per minute. |
| `SyntaxError` | Usually caused by a missing quotation mark `"`, bracket `)` or comma. Check the line shown in the error. |
| Unexpected behaviour | Select **Runtime → Restart session**, then run all cells from the top. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |